In [ ]:
#| default_exp support

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

Check and install kernel support in a selected Python environment.

In [ ]:
#| export
"Probe and install kernel support into a project's Python environment."

"Probe and install kernel support into a project's Python environment."

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import os, shutil, subprocess, sys, threading, time

In [ ]:
#| export
from pathlib import Path

In [ ]:
#| export
#: This interpreter's minor version. A kernel may borrow this `sys.path` only when it matches.
HOST_PY = tuple(sys.version_info[:2])

In [ ]:
#| export
INSPECTOR, INSPECTOR_TTL = ('dhrishti',), 90
KERNEL_PACKAGES = ('ipykernel', 'ipymini', 'kernmini', 'fastcore', 'microio')

`KERNEL_PACKAGES` lists the required distributions. `as_installed` adds versions from the host environment.

In [ ]:
HOST_PY, INSPECTOR, KERNEL_PACKAGES

((3, 11),
 ('dhrishti',),
 ('ipykernel', 'ipymini', 'kernmini', 'fastcore', 'microio'))

In [ ]:
#| export
_INSPECT = {}

In [ ]:
#| export
_locks, _locks_lock = {}, threading.Lock()

In [ ]:
#| export
from kunda.pythons import BUNDLE_ONLY, clean_env, strip_bundle

In [ ]:
#| export
def work_dir(cwd=None):
    "`cwd` as a string, falling back to a writable directory when the process inherited `/` or a read-only bundle."
    if cwd: return str(cwd)
    here = os.getcwd()
    if not getattr(sys, 'frozen', False): return here
    return here if here != os.sep and os.access(here, os.W_OK) else str(Path.home())

`work_dir` returns a supplied directory as text. A missing directory uses the current working directory.

In [ ]:
work_dir(Path('/repo/notebooks'))

'/repo/notebooks'

In [ ]:
#| hide
here = os.getcwd()
try:
    sys.frozen = True; os.chdir(os.sep)
    test_eq(work_dir(), str(Path.home()))   # `/` is what a double-clicked app inherits
    test_eq(work_dir('/repo'), '/repo')     # a named directory is still kept
finally: os.chdir(here); del sys.frozen
test_eq(work_dir(), here)

In [ ]:
#| export
def support_paths():
    "The `sys.path` entries `bootstrap_src` hands a kernel to borrow from."
    return [p for p in sys.path if p and os.path.isabs(p) and os.path.exists(p)]

`support_paths` returns existing absolute import paths. Relative and missing paths are excluded.

In [ ]:
#| export
def _lock_for(python):
    "One lock per interpreter: two installers in one venv is a race, and three surfaces offer it."
    with _locks_lock: return _locks.setdefault(str(python), threading.Lock())

Each interpreter path has one shared probe lock.

In [ ]:
test_is(_lock_for('/a/.venv/bin/python'), _lock_for('/a/.venv/bin/python'))
assert _lock_for('/a/.venv/bin/python') is not _lock_for('/b/.venv/bin/python')

In [ ]:
#| export
class _Failed:
    "What a command that never started looks like, so a probe reports instead of raising."
    returncode, stdout = 127, ''
    def __init__(self, err): self.stderr = err

def _run(args, timeout=180):
    "Run a command and return its result; missing interpreter and timeout are returned as answers, not raised."
    try:
        return subprocess.run([str(x) for x in args], env=clean_env(), text=True,
            encoding='utf-8', errors='replace', capture_output=True, timeout=timeout)
    except subprocess.TimeoutExpired: return _Failed(f'timed out after {timeout}s')
    except OSError as e: return _Failed(str(e))

`_run` always returns `returncode`, `stdout`, and `stderr`. Process-start failures become a non-zero result.

In [ ]:
p = _run(['/definitely/not/a/python', '-c', 'pass'])
p.returncode, p.stderr

(127, "[Errno 2] No such file or directory: '/definitely/not/a/python'")

In [ ]:
#| hide
p = _run([sys.executable, '-c', 'import time; time.sleep(5)'], timeout=1)
test_eq((p.returncode, p.stdout), (127, ''))
test_eq(p.stderr, 'timed out after 1s')

In [ ]:
#| export
def _last_line(text):
    "The line a traceback ends on. numpy's ABI failure wraps its own in eighteen lines of advice."
    return next((l for l in reversed((text or '').strip().splitlines()) if l.strip()), '').strip()

`_last_line` returns the last non-empty line, or an empty string.

In [ ]:
_last_line('Traceback (most recent call last):\n  File "<string>", line 1\nImportError: numpy ABI\n\n')

'ImportError: numpy ABI'

In [ ]:
#| hide
test_eq(_last_line(''), '')
test_eq(_last_line(None), '')
test_eq(_last_line('  only line  '), 'only line')

In [ ]:
#| export
def _probe(python, module, paths=()):
    "Whether `module` imports in `python`, with its version or the failure."
    pre = (f'import sys\nif tuple(sys.version_info[:2]) == {HOST_PY!r}: '
           f'sys.path.extend({list(paths)!r})\n') if paths else ''
    p = _run([python, '-c', f'{pre}import {module}; print(getattr({module}, "__version__", "installed"))'], 15)
    return {'available': p.returncode == 0,
        'version': p.stdout.strip() if p.returncode == 0 else '',
        'error': _last_line(p.stderr or p.stdout) if p.returncode else ''}

A module probe returns `available`, `version`, and `error`.

In [ ]:
tmp = TemporaryDirectory(); d = Path(tmp.name)
(d/'borrowed.py').write_text('__version__ = "1.2"')
_probe(sys.executable, 'borrowed', paths=[str(d)])

{'available': True, 'version': '1.2', 'error': ''}

In [ ]:
#| hide
miss = _probe(sys.executable, 'borrowed')
test_eq((miss['available'], miss['version']), (False, ''))
assert 'ModuleNotFoundError' in miss['error']
test_eq(_probe(sys.executable, 'os')['version'], 'installed')

In [ ]:
#| export
def kernel_support(python):
    "Importability of both supported kernel launchers in ``python``."
    python = str(Path(python).expanduser())
    return {m: _probe(python, m) for m in ('ipykernel', 'ipymini')}

`kernel_support` probes both kernel launchers in the selected interpreter.

In [ ]:
ok = kernel_support(sys.executable)['ipymini']
gone = kernel_support('/definitely/not/a/python')['ipymini']
ok, gone

({'available': True, 'version': '0.1.21', 'error': ''},
 {'available': False,
  'version': '',
  'error': "[Errno 2] No such file or directory: '/definitely/not/a/python'"})

In [ ]:
#| export
def inspector_support(python, refresh=False):
    "Check whether `python` can import the live-variable inspector, returning what blocked it if not."
    python = str(Path(python).expanduser())
    hit = _INSPECT.get(python)
    if not refresh and hit and time.time() - hit[0] < INSPECTOR_TTL: return hit[1]
    state = _probe(python, 'dhrishti.serving', support_paths())
    _INSPECT[python] = (time.time(), state)
    return state

IMPORT_FAULTS = ('ModuleNotFoundError', 'ImportError', 'No module named')

def import_failure(err):
    "Whether bootstrap failed on an inspector import."
    return any(s in (err or '') for s in IMPORT_FAULTS)

`inspector_support` caches one result per interpreter for `INSPECTOR_TTL` seconds. `import_failure` recognizes a missing or broken inspector import.

In [ ]:
inspector_support(sys.executable), import_failure("ModuleNotFoundError: No module named 'dhrishti'")

{'available': False,
 'version': '',
 'error': "ModuleNotFoundError: No module named 'dhrishti'"}

In [ ]:
#| hide
s = inspector_support(sys.executable)
test_is(inspector_support(sys.executable), s)
_INSPECT[sys.executable] = (time.time() - INSPECTOR_TTL - 1, {'available': 'stale'})
test_eq(set(inspector_support(sys.executable)), {'available', 'version', 'error'})   # expired, asked again

test_eq(import_failure(None), False)
test_eq(import_failure(''), False)
numpy_abi = ('RuntimeError: module compiled against API version 0x10 but this version of numpy is 0xf\n'
             'Consider using a different interpreter, or reinstalling numpy, or ...\n'
             'ImportError: numpy.core.multiarray failed to import')
assert import_failure(numpy_abi)

In [ ]:
#| export
def installable(python):
    "Whether anything may install into `python`. Never this one, and never inside a signed bundle."
    if not python: return False
    p = os.path.abspath(str(python))
    if p == os.path.abspath(sys.executable): return False
    return '/Contents/Resources/' not in p and '/Contents/MacOS/' not in p

`installable` rejects a missing path, the current interpreter, and interpreters inside a macOS application bundle.

In [ ]:
installable(sys.executable), installable('/repo/.venv/bin/python')

(False, True)

In [ ]:
#| export
def as_installed(names=KERNEL_PACKAGES):
    "Pin `names` to the versions this process runs; leave unpinned what it doesn't have, so installs stay compatible with the host."
    from importlib.metadata import PackageNotFoundError, version
    out = []
    for name in names:
        try: out.append(f'{name}=={version(name)}')
        except (PackageNotFoundError, ValueError, OSError): out.append(name)
    return out

`as_installed` pins names to versions installed in the host process.

In [ ]:
as_installed(['fastcore', 'ipymini', 'not_a_real_package'])

['fastcore==2.2.19', 'ipymini==0.1.21', 'not_a_real_package']

In [ ]:
#| export
def _attempts(python, packages):
    "Installer commands to try, in order, for environments that intentionally lack pip."
    out = []
    if uv := shutil.which('uv'): out.append([uv, 'pip', 'install', '--python', python, *packages])
    out.append([python, '-m', 'pip', 'install', *packages])
    if not uv and _run([python, '-m', 'ensurepip', '--upgrade']).returncode == 0:
        out.insert(0, [python, '-m', 'pip', 'install', *packages])
    return out

`_attempts` tries `uv` when available and always includes `python -m pip install`.

In [ ]:
_attempts('/nowhere/bin/python', ['ipymini==0.1.21'])[-1]

['/nowhere/bin/python', '-m', 'pip', 'install', 'ipymini==0.1.21']

In [ ]:
#| hide
cmds = _attempts('/nowhere/bin/python', ['a', 'b'])
test_eq(cmds[-1], ['/nowhere/bin/python', '-m', 'pip', 'install', 'a', 'b'])
if shutil.which('uv'): test_eq(cmds[0][1:5], ['pip', 'install', '--python', '/nowhere/bin/python'])

`_log` records the command, return code, and bounded output.

In [ ]:
#| export
def _log(command, p): return {'command': command, 'returncode': p.returncode,'output': (p.stdout + '\n' + p.stderr).strip()[-8000:]}
def _detail(logs): return next((a['output'] for a in reversed(logs) if a['output']), 'no installer answered')

In [ ]:
#| hide
logs = [_log(['pip'], _Failed('')), _log(['uv'], _Failed('no interpreter'))]
test_eq(logs[0], {'command': ['pip'], 'returncode': 127, 'output': ''})
test_eq(_detail(logs), 'no interpreter')
test_eq(_detail(logs[:1]), 'no installer answered')
test_eq(_log(['pip'], _Failed('x' * 9000))['output'], 'x' * 8000)

In [ ]:
#| export
def install_kernel_support(python, packages=None):
    "Install kernel packages via uv or pip; verify the import works after, so failures name the package not the installer."
    python = str(Path(python).expanduser().absolute())
    packages, logs, unimportable = list(packages or as_installed()), [], ''
    with _lock_for(python):
        for command in _attempts(python, packages):
            p = _run(command, 600)
            logs.append(_log(command, p))
            if p.returncode == 0:
                support = kernel_support(python)
                if all(x['available'] for x in support.values()): return {'ok': True, 'python': python, 'support': support, 'attempts': logs}
                unimportable = '; '.join(f"{name} installed but does not import ({x['error']})" for name, x in support.items() if not x['available'])
    raise RuntimeError(f'kernel support installation failed: {unimportable or _detail(logs)}')

`install_kernel_support` succeeds only when installation completes and both launchers import.

In [ ]:
#| hide
# No installer can start against a path that is not an interpreter, so this installs nothing.
test_fail(install_kernel_support, args=['/nowhere/bin/python'], contains='kernel support installation failed')

In [ ]:
#| export
def install_inspector_support(python, packages=INSPECTOR):
    "Give `python` its own copy of what the inspector imports, then ask it again."
    python = str(Path(python).expanduser().absolute())
    logs = []
    with _lock_for(python):
        for command in _attempts(python, packages):
            p = _run(command, 600)
            logs.append(_log(command, p))
            if p.returncode != 0: continue
            state = inspector_support(python, refresh=True)
            if state['available']: return {'ok': True, 'python': python, 'inspector': state, 'attempts': logs}
            raise RuntimeError(f'{" and ".join(packages)} are installed, and the inspector still will not import: {state["error"]}')
    raise RuntimeError(f'could not install {" and ".join(packages)}: {_detail(logs)}')

`install_inspector_support` installs Dhrishti and probes it again.

In [ ]:
#| hide
tmp.cleanup()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()